# Loki no-LGTM assessment

Assesses Loki **without using the LGTM stack as an instrument** -- kube API state,
events, PVCs, and direct HTTP `/ready` probes only. The realistic tool for when
Loki is unhappy and therefore can't answer questions about itself (its own metrics
flow through the same pipeline and may be dark or lying). The query smoke test at
the end touches Loki, but as the system under test, never as an instrument.

Normal-times status: [loki-health.ipynb](loki-health.ipynb); capacity:
[loki-usage.ipynb](loki-usage.ipynb). Design: [tiles#644](https://github.com/symmatree/tiles/issues/644).

Run from `notebooks/`: `jupyter nbconvert --to notebook --execute --inplace loki-nolgtm.ipynb`

In [1]:
namespace = "loki"
loki_url = "http://loki.loki.svc:3100"   # smoke-test target only
tenant = "tiles"                         # X-Scope-OrgID for the smoke test
capture_file = ""      # replay a prior raw capture JSON instead of querying live
output_dir = ""        # if set: write raw capture + agent stats there
debug = False

In [2]:
import json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import nb_capture as nbc

NOTEBOOK = 'loki-nolgtm'
cap = nbc.Capture(replay_file=capture_file, namespace=namespace,
                  loki_url=loki_url, tenant=tenant)
namespace, tenant = cap.meta['namespace'], cap.meta['tenant']
kube = nbc.Kube(cap)
print(f"namespace: {namespace}  "
      f"{'REPLAY of ' + cap.run_at if cap.replay else 'live'}")

namespace: loki  live


In [3]:
# --- ASSUMPTIONS: only that the kube API answers (Kube.get raises otherwise).
# Deliberately nothing else -- every other system here is under test, not trusted.
ns = kube.get('namespace', 'namespace', namespace)
print(f"kube API: ok; namespace {namespace} is {ns['status']['phase']}")

kube API: ok; namespace loki is Active


In [4]:
# --- PODS: phase, readiness, restarts, last terminations (kube API view) ---
pods_raw = kube.get('pods', 'pods', '-n', namespace)
rows = []
for p in pods_raw['items']:
    cs = p['status'].get('containerStatuses', [])
    term = next(((c['lastState'].get('terminated') or {}) for c in cs
                 if c['lastState'].get('terminated')), {})
    rows.append({'pod': p['metadata']['name'],
                 'phase': p['status'].get('phase', '?'),
                 'ready': f"{sum(c['ready'] for c in cs)}/{len(cs)}",
                 'restarts': sum(c['restartCount'] for c in cs),
                 'node': p['spec'].get('nodeName', ''),
                 'last_term': f"{term.get('reason', '')} {(term.get('finishedAt') or '')[:16]}".strip()})
pods_df = pd.DataFrame(rows).set_index('pod').sort_index()
ready_n = pods_df['ready'].str.split('/', expand=True)
not_ready = sorted(pods_df.index[(pods_df['phase'] != 'Running')
                                 | (ready_n[0] != ready_n[1])])
print(pods_df.to_string())
print(f"not ready: {not_ready or 'none'}")

                     phase ready  restarts        node               last_term
pod                                                                           
loki-0             Running   2/2         5  tiles-wk-3  Error 2026-07-25T05:56
loki-canary-clnfq  Running   1/1         1      lancer  ContainerStatusUnknown
loki-canary-dx65t  Running   1/1         0  tiles-wk-1                        
loki-canary-kvrp7  Running   1/1         0  tiles-wk-2                        
loki-canary-sgs27  Running   1/1         0  tiles-wk-3                        
not ready: none


In [5]:
# --- EVENTS: Warning events in the namespace (whatever etcd still holds) ---
ev = kube.get('events', 'events', '-n', namespace)

def ev_time(e):
    return e.get('lastTimestamp') or e.get('eventTime') or e['metadata']['creationTimestamp']

warn = sorted((e for e in ev['items'] if e.get('type') == 'Warning'),
              key=ev_time, reverse=True)
warn_reasons = dict(Counter(e.get('reason', '?') for e in warn))
print(f"{len(ev['items'])} events retained, {len(warn)} warnings; by reason: {warn_reasons or 'none'}")
for e in warn[:15]:
    print(f"  {ev_time(e)[:19]} x{e.get('count') or 1:<3} {e.get('reason', '?'):<18} "
          f"{e['involvedObject'].get('name', '')[:38]:<38} {e.get('message', '')[:100]}")

0 events retained, 0 warnings; by reason: none


In [6]:
# --- DIRECT /ready PROBES: each component service, no metrics involved.
# SingleBinary Loki exposes one http-metrics port (3100) with /ready; the canary
# service (3500) is probed for reachability of its /metrics (no /ready endpoint).
svcs = kube.get('services', 'services', '-n', namespace)
probe = {}
for s in svcs['items']:
    nm = s['metadata']['name']
    if nm.endswith('-headless') or nm.endswith('-memberlist'):
        continue
    http_port = next((p['port'] for p in s['spec']['ports']
                      if p.get('name') == 'http-metrics' or p['port'] in (3100, 3500)), None)
    if http_port is None:
        continue
    path = '/ready' if http_port == 3100 else '/metrics'
    rec = cap.http(f'ready: {nm}', f'http://{nm}.{namespace}.svc:{http_port}{path}', timeout=5)
    body = (rec.get('text') or json.dumps(rec.get('json', ''))).strip()[:40]
    probe[nm] = rec.get('error') or f"{rec.get('status')} {body}"
    print(f"  {nm:<24}:{http_port}{path:<9} {probe[nm]}")
ready_failed = sorted(nm for nm, r in probe.items() if not str(r).startswith('200'))
print(f"failed probes: {ready_failed or 'none'}")

  loki                    :3100/ready    200 ready
  loki-canary             :3500/metrics  200 # HELP deprecated_flags_inuse_total The 
failed probes: none


In [7]:
# --- PVCs (loki-loki-data = NFS RWX chunks/rules; storage-loki-0 = local-path WAL) ---
pvcs = kube.get('pvcs', 'pvc', '-n', namespace)
pvc_phase = {p['metadata']['name']: p['status']['phase'] for p in pvcs['items']}
pvc_not_bound = sorted(n for n, ph in pvc_phase.items() if ph != 'Bound')
print(f"{len(pvc_phase)} PVCs: {pvc_phase}")
print(f"not bound: {pvc_not_bound or 'none'}")

2 PVCs: {'loki-loki-data': 'Bound', 'storage-loki-0': 'Bound'}
not bound: none


In [8]:
# --- SMOKE TEST (system under test, not an instrument): does the read path answer? ---
smoke = {}
for nm, path, params in [('ready', '/ready', None),
                         ('labels', '/loki/api/v1/labels', None),
                         ('query', '/loki/api/v1/query', {'query': 'vector(1)'})]:
    hdr = {'X-Scope-OrgID': tenant} if nm != 'ready' else None
    rec = cap.http(f'smoke: {nm}', f"{cap.meta['loki_url']}{path}", params=params,
                   headers=hdr, timeout=10)
    smoke[nm] = rec.get('error') or rec.get('status')
    print(f"  {nm:<8} {smoke[nm]}")
read_path_ok = smoke.get('labels') == 200
print(f"read path answers: {read_path_ok}")

  ready    200
  labels   200


  query    200
read path answers: True


In [9]:
# --- SUMMARY ---
findings = []
if not_ready:
    findings.append(f"pods not ready: {not_ready}")
if ready_failed:
    findings.append(f"/ready probes failing: {ready_failed}")
if pvc_not_bound:
    findings.append(f"PVCs not bound: {pvc_not_bound}")
if not read_path_ok:
    findings.append(f"read path NOT answering: {smoke}")
bad_reasons = {r: n for r, n in warn_reasons.items()
               if r in ('OOMKilling', 'BackOff', 'CrashLoopBackOff', 'FailedScheduling',
                        'FailedMount', 'Unhealthy', 'Evicted')}
if bad_reasons:
    findings.append(f"warning events of concern: {bad_reasons}")

print("=" * 60)
print("LOKI NO-LGTM SUMMARY")
print("=" * 60)
print(f"run:      {cap.run_at}  ({cap.mode})")
print(f"scope:    namespace {namespace}, kube API + direct HTTP only")
print()
if findings:
    print("Findings:")
    for f in findings:
        print(f"  ! {f}")
else:
    print(f"Findings: none -- {len(pods_df)} pods running/ready, "
          f"{len(probe)} probes green, PVCs bound, read path answers")

agent_stats = {
    'notebook': f'{NOTEBOOK}.ipynb',
    'run_at': cap.run_at,
    'mode': cap.mode,
    'namespace': namespace,
    'findings': findings,
    'pods': {'total': len(pods_df), 'not_ready': not_ready,
             'restarts_by_pod': {p: int(n) for p, n in pods_df['restarts'].items() if n > 0},
             'last_term_by_pod': {p: t for p, t in pods_df['last_term'].items() if t}},
    'events': {'warning_count': len(warn), 'warning_reasons': warn_reasons},
    'probes': probe,
    'pvcs': {'phase': pvc_phase, 'not_bound': pvc_not_bound},
    'smoke': smoke,
    'read_path_ok': read_path_ok,
}
print()
print(json.dumps(agent_stats, indent=1))

if output_dir:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cap.save(out / f'{NOTEBOOK}.capture.json')
    (out / f'{NOTEBOOK}.stats.json').write_text(json.dumps(agent_stats, indent=2))
    print(f"wrote {out / f'{NOTEBOOK}.capture.json'} and {out / f'{NOTEBOOK}.stats.json'}")

LOKI NO-LGTM SUMMARY
run:      2026-07-29T16:46:46.194384+00:00  (live)
scope:    namespace loki, kube API + direct HTTP only

Findings: none -- 5 pods running/ready, 2 probes green, PVCs bound, read path answers

{
 "notebook": "loki-nolgtm.ipynb",
 "run_at": "2026-07-29T16:46:46.194384+00:00",
 "mode": "live",
 "namespace": "loki",
 "findings": [],
 "pods": {
  "total": 5,
  "not_ready": [],
  "restarts_by_pod": {
   "loki-0": 5,
   "loki-canary-clnfq": 1
  },
  "last_term_by_pod": {
   "loki-0": "Error 2026-07-25T05:56",
   "loki-canary-clnfq": "ContainerStatusUnknown"
  }
 },
 "events": {
  "warning_count": 0,
  "warning_reasons": {}
 },
 "probes": {
  "loki": "200 ready",
  "loki-canary": "200 # HELP deprecated_flags_inuse_total The "
 },
 "pvcs": {
  "phase": {
   "loki-loki-data": "Bound",
   "storage-loki-0": "Bound"
  },
  "not_bound": []
 },
 "smoke": {
  "ready": 200,
  "labels": 200,
  "query": 200
 },
 "read_path_ok": true
}
